In [1]:
# pothole_classifier.py
import json
from pathlib import Path
from typing import Union, List, Sequence
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image


class PotholeClassifier:
    """
    Usage:
        clf = PotholeClassifier(r"path/to/pothole_model_v3_export")
        r = clf.predict("some_image.jpg")
        # -> {"pothole_probability": 0.87, "is_pothole": True, "threshold": 0.5}
    """

    def __init__(self, export_dir: Union[str, Path],
                 device: Union[str, torch.device, None] = None,
                 threshold: Union[float, None] = None):
        export_dir = Path(export_dir)
        with open(export_dir / "config.json", "r", encoding="utf-8") as f:
            self.config = json.load(f)

        self.device = torch.device(device) if device else \
                      torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.pothole_idx = int(self.config["pothole_idx"])
        self.threshold = float(
            threshold if threshold is not None else self.config["default_threshold"]
        )

        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, self.config["num_classes"])
        sd = torch.load(export_dir / "weights.pth", map_location=self.device)
        model.load_state_dict(sd)
        self.model = model.to(self.device).eval()

        size = int(self.config["input_size"])
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(self.config["normalize_mean"],
                                 self.config["normalize_std"]),
        ])

    @torch.no_grad()
    def predict(self, image):
        single = not isinstance(image, (list, tuple))
        imgs = [image] if single else list(image)

        tensors = []
        for im in imgs:
            if isinstance(im, (str, Path)):
                im = Image.open(im).convert("RGB")
            elif isinstance(im, Image.Image):
                im = im.convert("RGB")
            else:
                raise TypeError(f"Unsupported input type: {type(im)}")
            tensors.append(self.transform(im))

        batch = torch.stack(tensors).to(self.device)
        probs = torch.softmax(self.model(batch), dim=1)[:, self.pothole_idx]
        probs = probs.cpu().numpy().tolist()

        results = [{
            "pothole_probability": float(p),
            "is_pothole": bool(p >= self.threshold),
            "threshold": self.threshold,
        } for p in probs]
        return results[0] if single else results

In [2]:
## classifier use guidence

from pothole_classifier import PotholeClassifier

clf = PotholeClassifier(r"your file\pothole_model_v3_export")

print(clf.predict(r"\some\image.jpg"))

results = clf.predict([r"a.jpg", r"b.jpg", r"c.jpg"])
for r in results:
    print(r)

clf2 = PotholeClassifier(
    r"Yourfile/pothole_model_v3_export",
    device="cpu",
    threshold=0.60,
)

ModuleNotFoundError: No module named 'pothole_classifier'

In [3]:
import torch
from pathlib import Path

export_dir = Path(r"D:\Big_Data\pothole_model_v3_export")
export_dir.mkdir(parents=True, exist_ok=True)

# make sure model is in eval mode
model_v3.eval()

# create one dummy input with the same size as inference input
example = torch.randn(1, 3, 224, 224, device=device)

# trace and save
with torch.no_grad():
    traced_model = torch.jit.trace(model_v3, example)
    traced_model.save(str(export_dir / "model_v3_traced.pt"))

print("Saved traced model to:")
print(export_dir / "model_v3_traced.pt")

NameError: name 'model_v3' is not defined